# EES 2110 — Time Series: Climate, CO₂, and Real-World 2 m Air Temperature

**Upgraded notebook.** This version keeps the core class exercises but removes the Google-Drive-only dependency and adds a robust real-world 2 m air-temperature workflow using the Open-Meteo Historical Weather API.

### What was fixed
- `temperature_2m` is downloaded directly from a maintained historical-weather API.
- No API key is required for ordinary non-commercial use.
- Dates and coordinates are easy to edit.
- HTTP errors, malformed responses, empty data, and missing values are checked explicitly.
- Downloaded data are cached to CSV so you can rerun plots offline after the first successful download.
- The original `climate_data.csv` workflow works in Jupyter **or** Google Colab.

> Internet access is required only for the first real-world weather download.


In [ ]:
import sys
import json
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)


## Part A — Load the course climate/CO₂ CSV

Put `climate_data.csv` in the same folder as this notebook. In Colab, if it is not present, the next cell offers a file-upload dialog. This replaces the original hard-coded `/content/drive/...` path.


In [ ]:
def load_course_csv(filename="climate_data.csv"):
    path = Path(filename)
    if path.exists():
        print(f"Loading {path.resolve()}")
        return pd.read_csv(path)

    # Colab fallback
    try:
        from google.colab import files
        print(f"{filename} was not found. Please upload it now.")
        uploaded = files.upload()
        if filename not in uploaded:
            if len(uploaded) == 1:
                filename = next(iter(uploaded))
            else:
                raise FileNotFoundError("Please upload climate_data.csv")
        return pd.read_csv(filename)
    except ImportError:
        raise FileNotFoundError(
            f"Could not find {filename!r}. Put it in the same folder as this notebook "
            "and rerun this cell."
        )

# Uncomment when you have climate_data.csv:
# df_climate = load_course_csv()
# display(df_climate.head())


In [ ]:
# After loading df_climate, use this cell:
# required = {"T_column1", "T_column2", "CO2_column1", "CO2_column2"}
# missing = required - set(df_climate.columns)
# if missing:
#     raise ValueError(f"Missing expected columns: {sorted(missing)}")
#
# T = df_climate[["T_column1", "T_column2"]].to_numpy(dtype=float)
# CO2 = df_climate[["CO2_column1", "CO2_column2"]].to_numpy(dtype=float)
# print("Temperature shape:", T.shape)
# print("CO2 shape:", CO2.shape)
# print("Mean temperature anomaly:", np.nanmean(T[:, 1]))


## Part B — Fixed real-world 2 m air temperature

The function below requests **hourly air temperature at 2 m above ground**. Change the location and dates in the settings cell.

For long climate records, `model="era5"` is a good consistent choice. The API documentation describes ERA5 as available from 1940 with a short publication delay.


In [ ]:
# EDIT THESE SETTINGS
LOCATION_NAME = "Nashville, Tennessee"
LATITUDE = 36.1627
LONGITUDE = -86.7816
START_DATE = "2025-01-01"
END_DATE = "2025-12-31"
MODEL = "era5"          # consistent reanalysis for climate/time-series work
TIMEZONE = "auto"
TEMPERATURE_UNIT = "celsius"

CACHE_FILE = Path(
    f"temperature_2m_{LATITUDE:.3f}_{LONGITUDE:.3f}_{START_DATE}_{END_DATE}_{MODEL}.csv"
)
CACHE_FILE


In [ ]:
def download_temperature_2m(
    latitude, longitude, start_date, end_date,
    timezone="auto", temperature_unit="celsius", model="era5",
    timeout=30
):
    """Download hourly 2 m air temperature from Open-Meteo Historical Weather API."""
    base_url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": float(latitude),
        "longitude": float(longitude),
        "start_date": str(start_date),
        "end_date": str(end_date),
        "hourly": "temperature_2m",
        "temperature_unit": temperature_unit,
        "timezone": timezone,
        "models": model,
    }
    url = base_url + "?" + urlencode(params)
    request = Request(url, headers={"User-Agent": "EES2110-Jupyter/1.0"})

    try:
        with urlopen(request, timeout=timeout) as response:
            payload = json.loads(response.read().decode("utf-8"))
    except HTTPError as e:
        detail = e.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Weather API HTTP {e.code}: {detail}") from e
    except URLError as e:
        raise RuntimeError(
            "Could not reach the weather API. Check your internet connection, then rerun. "
            f"Original error: {e.reason}"
        ) from e
    except json.JSONDecodeError as e:
        raise RuntimeError("Weather service returned invalid JSON.") from e

    if payload.get("error"):
        raise RuntimeError("Weather API error: " + str(payload.get("reason", payload)))

    hourly = payload.get("hourly", {})
    times = hourly.get("time")
    temps = hourly.get("temperature_2m")
    if not times or temps is None:
        raise RuntimeError(
            "The API response did not contain hourly temperature_2m data. "
            "Try an earlier end date or check the selected model/date range."
        )
    if len(times) != len(temps):
        raise RuntimeError("API returned different numbers of timestamps and temperatures.")

    df = pd.DataFrame({
        "time": pd.to_datetime(times, errors="coerce"),
        "temperature_2m": pd.to_numeric(temps, errors="coerce"),
    }).dropna(subset=["time"]).sort_values("time").reset_index(drop=True)

    if df.empty or df["temperature_2m"].notna().sum() == 0:
        raise RuntimeError("No valid 2 m temperature values were returned.")

    unit = payload.get("hourly_units", {}).get("temperature_2m", "°C")
    df.attrs["temperature_unit"] = unit
    df.attrs["latitude"] = payload.get("latitude")
    df.attrs["longitude"] = payload.get("longitude")
    df.attrs["elevation"] = payload.get("elevation")
    df.attrs["timezone"] = payload.get("timezone")
    return df, url


def get_temperature_2m(cache_file=CACHE_FILE, force_download=False):
    """Use cached CSV when available; otherwise download and cache the data."""
    cache_file = Path(cache_file)
    if cache_file.exists() and not force_download:
        print(f"Using cached data: {cache_file}")
        df = pd.read_csv(cache_file, parse_dates=["time"])
        return df, None

    df, url = download_temperature_2m(
        LATITUDE, LONGITUDE, START_DATE, END_DATE,
        timezone=TIMEZONE, temperature_unit=TEMPERATURE_UNIT, model=MODEL
    )
    df.to_csv(cache_file, index=False)
    print(f"Downloaded {len(df):,} hourly values and saved {cache_file}")
    return df, url


In [ ]:
# RUN THIS CELL
weather, request_url = get_temperature_2m(force_download=False)

print(weather.head())
print()
print("Rows:", len(weather))
print("Valid temperatures:", weather["temperature_2m"].notna().sum())
print("Time range:", weather["time"].min(), "to", weather["time"].max())
print("Temperature range:", weather["temperature_2m"].min(), "to", weather["temperature_2m"].max())
if request_url:
    print()
    print("Request used:", request_url)


In [ ]:
# Plot the hourly real-world 2 m air temperature
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(weather["time"], weather["temperature_2m"], linewidth=0.7)
ax.set_xlabel("Date")
ax.set_ylabel("2 m air temperature (°C)")
ax.set_title(f"Hourly 2 m Air Temperature — {LOCATION_NAME}")
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()


In [ ]:
# Convert hourly data to daily mean/min/max for a cleaner time series
daily = (
    weather.set_index("time")["temperature_2m"]
    .resample("D")
    .agg(["mean", "min", "max"])
    .dropna(how="all")
)

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(daily.index, daily["mean"], label="Daily mean", linewidth=1.2)
ax.fill_between(daily.index, daily["min"], daily["max"], alpha=0.18, label="Daily min–max")
ax.set_xlabel("Date")
ax.set_ylabel("2 m air temperature (°C)")
ax.set_title(f"Daily 2 m Air Temperature — {LOCATION_NAME}")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

display(daily.head())


## Part C — Time-series exercises (corrected versions)


In [ ]:
# Exercise: sum integers 1 through 2020
total = 0
for i in range(1, 2021):
    total += i
print(total)


In [ ]:
# Exercise: y = sin(x), -pi to +pi
x = np.arange(-np.pi, np.pi + 0.1, 0.1)
y = np.sin(x)
plt.figure(figsize=(7, 4))
plt.plot(x, y)
plt.xlabel("x")
plt.ylabel("sin(x)")
plt.title("y = sin(x)")
plt.grid(alpha=0.25)
plt.show()


In [ ]:
# Exercise: sine(x) and 5*cos(x) on two y-axes
x = np.arange(-np.pi, np.pi + 0.1, 0.1)
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(x, np.sin(x))
ax1.set_xlabel("x")
ax1.set_ylabel("sin(x)")
ax2 = ax1.twinx()
ax2.plot(x, 5*np.cos(x), linestyle="--")
ax2.set_ylabel("5 cos(x)")
ax1.set_title("sin(x) and 5 cos(x)")
fig.tight_layout()
plt.show()


In [ ]:
# Corrected buggy cosine example
x = np.arange(-2*np.pi, np.pi, 0.1)
f = np.cos(3*x)
plt.figure(figsize=(7, 4))
plt.plot(x, f)
plt.xlabel("x")
plt.ylabel("f")
plt.title("f(x) = cos(3x)")
plt.grid(alpha=0.25)
plt.show()


In [ ]:
def smooth_timeseries(d):
    """3-point weighted smoother: 0.25*previous + 0.5*current + 0.25*next."""
    d = np.asarray(d, dtype=float)
    if len(d) < 3:
        raise ValueError("Need at least 3 points to smooth the time series.")
    return 0.25*d[:-2] + 0.5*d[1:-1] + 0.25*d[2:]

# Apply smoothing to the daily mean real-world temperature
temp = daily["mean"].to_numpy()
temp_smooth = smooth_timeseries(temp)
time_smooth = daily.index[1:-1]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily.index, temp, label="Daily mean", alpha=0.65)
ax.plot(time_smooth, temp_smooth, label="3-point smoothed", linewidth=1.6)
ax.set_xlabel("Date")
ax.set_ylabel("2 m air temperature (°C)")
ax.set_title(f"Original and Smoothed Temperature — {LOCATION_NAME}")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()


## Troubleshooting the real-world download

If the download cell fails:

1. Confirm the notebook has internet access.
2. Use ISO dates like `2025-01-01`.
3. For ERA5, avoid the most recent few days because reanalysis is published with a delay.
4. Keep latitude in `[-90, 90]` and longitude in `[-180, 180]`.
5. Delete the cache CSV or run `get_temperature_2m(force_download=True)` if you change settings but reuse the same filename.

The downloader prints a useful error instead of silently producing an empty plot.
